<a href="https://colab.research.google.com/github/anubagar/anubagar/blob/main/Copy_of_LoRA_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LoRA Fine-Tuning: From Basics to Practical PEFT

## Classroom Lecture + Hands-on Notebook

This notebook explains **LoRA (Low-Rank Adaptation)** from first principles and then implements it on a real Transformer model.

### Contents

1. What is fine-tuning?
2. Why full fine-tuning is expensive
3. Parameter-efficient fine-tuning
4. LoRA intuition
5. LoRA mathematics
6. Matrix dimensions
7. Why the update is low rank
8. LoRA initialization
9. The role of rank $r$
10. The role of scaling factor $\alpha$
11. LoRA dropout
12. Manual LoRA implementation in PyTorch
13. Gradient-flow demonstration
14. Parameter-count comparison
15. Where LoRA is inserted in a Transformer
16. Hugging Face PEFT implementation
17. Train and evaluate a real model
18. Save and reload an adapter
19. Merge LoRA with the base model
20. Experiments and classroom questions

> **Important:** All mathematical expressions in this notebook use standard Jupyter/MathJax syntax: inline math uses `$...$` and display equations use `$$...$$`.


# 1. What is Fine-Tuning?

A pretrained neural network has already learned useful representations from a large dataset.

Let its parameters be represented by a matrix $W$.

For an input vector $x$, a simple linear layer computes

$$
y = Wx.
$$

During **full fine-tuning**, we update the pretrained parameters using gradient descent:

$$
W \leftarrow W - \eta \nabla_W \mathcal{L},
$$

where:

- $\mathcal{L}$ is the training loss.
- $\eta$ is the learning rate.
- $\nabla_W \mathcal{L}$ is the gradient with respect to $W$.

Therefore, the original pretrained parameters themselves change during fine-tuning.


# 2. Why Can Full Fine-Tuning Be Expensive?

Suppose a layer contains

$$
W \in \mathbb{R}^{4096 \times 4096}.
$$

The number of parameters is

$$
4096 \times 4096 = 16,777,216.
$$

For a large Transformer, there can be many such matrices across many layers.

Full fine-tuning can therefore require:

- storing gradients for many parameters,
- storing optimizer states,
- updating a very large number of parameters,
- maintaining a separate model checkpoint for each task.

This motivates **parameter-efficient fine-tuning (PEFT)**.

The basic PEFT idea is:

$$
\boxed{\text{Keep most pretrained parameters frozen and train only a small set of parameters.}}
$$


# 3. What is LoRA?

LoRA stands for **Low-Rank Adaptation**.

Instead of directly learning a complete update matrix $\Delta W$, LoRA represents the update as a product of two smaller matrices:

$$
\Delta W = BA.
$$

The effective weight becomes

$$
W' = W + \Delta W.
$$

LoRA additionally uses a scaling factor:

$$
\boxed{
W' = W + \frac{\alpha}{r}BA
}
$$

where:

- $W$ is the frozen pretrained weight matrix.
- $A$ is a trainable low-rank matrix.
- $B$ is a trainable low-rank matrix.
- $r$ is the LoRA rank.
- $\alpha$ controls the scale of the LoRA update.

Only $A$ and $B$ are trained.

The original $W$ remains frozen.


# 4. Why Does LoRA Save Parameters?

Suppose

$$
W \in \mathbb{R}^{d_{out} \times d_{in}}.
$$

Full fine-tuning would learn

$$
d_{out}d_{in}
$$

parameters.

LoRA instead learns

$$
A \in \mathbb{R}^{r \times d_{in}}
$$

and

$$
B \in \mathbb{R}^{d_{out} \times r}.
$$

Therefore the number of trainable LoRA parameters is

$$
P_{LoRA}
=
rd_{in}+d_{out}r.
$$

Equivalently,

$$
\boxed{
P_{LoRA}=r(d_{in}+d_{out})
}
$$

Compare this with

$$
P_{Full}=d_{out}d_{in}.
$$

When $r$ is much smaller than $d_{in}$ and $d_{out}$, the LoRA parameter count can be dramatically smaller.


In [ ]:
def compare_parameters(d_in, d_out, ranks):
    full = d_in * d_out
    print(f"Full fine-tuning: {full:,} parameters\n")

    for r in ranks:
        lora = r * (d_in + d_out)
        percentage = 100 * lora / full
        print(
            f"r={r:>2}: "
            f"LoRA={lora:,} | "
            f"{percentage:.4f}% of full"
        )

compare_parameters(
    d_in=4096,
    d_out=4096,
    ranks=[1, 4, 8, 16, 32, 64]
)


Full fine-tuning: 16,777,216 parameters

r= 1: LoRA=8,192 | 0.0488% of full
r= 4: LoRA=32,768 | 0.1953% of full
r= 8: LoRA=65,536 | 0.3906% of full
r=16: LoRA=131,072 | 0.7812% of full
r=32: LoRA=262,144 | 1.5625% of full
r=64: LoRA=524,288 | 3.1250% of full


# 5. The Mathematics of the LoRA Forward Pass

The original linear layer is

$$
y = Wx.
$$

LoRA adds a second branch:

$$
y =
Wx +
\frac{\alpha}{r}BAx.
$$

Therefore,

$$
\boxed{
y =
Wx +
\frac{\alpha}{r}BAx
}
$$

The computation can be understood as two paths.

### Original path

$$
x \rightarrow W \rightarrow Wx
$$

### LoRA path

$$
x \rightarrow A \rightarrow Ax \rightarrow B \rightarrow BAx
$$

The two outputs are then added.

This is why LoRA can preserve the pretrained computation while learning a task-specific correction.


# 6. Checking the Matrix Dimensions

Assume

$$
x \in \mathbb{R}^{d_{in}},
$$

$$
A \in \mathbb{R}^{r \times d_{in}},
$$

and

$$
B \in \mathbb{R}^{d_{out} \times r}.
$$

First,

$$
Ax \in \mathbb{R}^{r}.
$$

Then,

$$
B(Ax) \in \mathbb{R}^{d_{out}}.
$$

Therefore,

$$
BA \in
\mathbb{R}^{d_{out}\times d_{in}}.
$$

This is exactly the same shape as $W$.

Hence,

$$
W + \frac{\alpha}{r}BA
$$

is mathematically valid.


In [ ]:
import torch

# Numerical shape check

d_in = 6
d_out = 4
r = 2

W = torch.randn(d_out, d_in)
A = torch.randn(r, d_in)
B = torch.randn(d_out, r)
x = torch.randn(d_in)

print("W shape :", W.shape)
print("A shape :", A.shape)
print("B shape :", B.shape)
print("BA shape:", (B @ A).shape)
print("x shape :", x.shape)
print("Output  :", ((W + B @ A) @ x).shape)

W shape : torch.Size([4, 6])
A shape : torch.Size([2, 6])
B shape : torch.Size([4, 2])
BA shape: torch.Size([4, 6])
x shape : torch.Size([6])
Output  : torch.Size([4])


# 7. Why Is the Update Called Low Rank?

LoRA defines

$$
\Delta W = BA.
$$

Because $A$ has only $r$ rows and $B$ has only $r$ columns,

$$
\operatorname{rank}(BA) \leq r.
$$

Therefore,

$$
\boxed{
\operatorname{rank}(\Delta W) \leq r
}
$$

This is the central mathematical idea behind LoRA.

### Important distinction

LoRA does **not** require

$$
\operatorname{rank}(W) = r.
$$

Instead, it constrains the **learned update**

$$
\Delta W
$$

to have rank at most $r$.

The original pretrained matrix $W$ can remain a full-rank matrix.


In [ ]:
# Verify the rank constraint

d_in = 10
d_out = 8
r = 2

A = torch.randn(r, d_in)
B = torch.randn(d_out, r)

delta_W = B @ A

print("Shape of Delta W:", delta_W.shape)
print("Numerical matrix rank:", torch.linalg.matrix_rank(delta_W).item())
print("Maximum possible rank:", r)


Shape of Delta W: torch.Size([8, 10])
Numerical matrix rank: 2
Maximum possible rank: 2


# 8. Why Two Smaller Matrices Instead of One?

LoRA creates a bottleneck:

$$
d_{in}
\rightarrow r
\rightarrow d_{out}.
$$

The first matrix $A$ compresses the input representation:

$$
x \rightarrow Ax.
$$

The second matrix $B$ expands it back:

$$
Ax \rightarrow B(Ax).
$$

If $r$ is small, the adaptation is forced to pass through a low-dimensional space.

For example, instead of learning a complete

$$
4096 \times 4096
$$

update, LoRA with $r=8$ learns

$$
8 \times 4096
$$

and

$$
4096 \times 8.
$$

The total is only

$$
65,536
$$

trainable parameters.


# 9. LoRA Initialization

A common initialization is:

$$
A \sim \text{small random values}
$$

and

$$
B=0.
$$

Initially,

$$
BA=0.
$$

Therefore,

$$
W' =
W+
\frac{\alpha}{r}(0)
=
W.
$$

So the LoRA module initially behaves like the original pretrained layer.

During training, $A$ and $B$ are updated and the LoRA branch gradually learns a task-specific correction.


In [ ]:
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, d_in, d_out, rank=2, alpha=4.0):
        super().__init__()

        # Simulated pretrained parameter
        self.weight = nn.Parameter(torch.randn(d_out, d_in))

        # Freeze pretrained weight
        self.weight.requires_grad = False

        self.rank = rank
        self.alpha = alpha

        # Trainable LoRA matrices
        self.A = nn.Parameter(
            torch.randn(rank, d_in) * 0.01
        )

        # Zero initialization
        self.B = nn.Parameter(
            torch.zeros(d_out, rank)
        )

    def forward(self, x):
        base_output = x @ self.weight.T

        lora_output = (
            self.alpha / self.rank
        ) * (x @ self.A.T) @ self.B.T

        return base_output + lora_output


layer = LoRALinear(
    d_in=6,
    d_out=4,
    rank=2,
    alpha=4
)

x = torch.randn(3, 6)

y = layer(x)

print("Output shape:", y.shape)

Output shape: torch.Size([3, 4])


# 10. Verifying the Initialization

Because $B=0$ initially,

$$
BA=0.
$$

Therefore the LoRA contribution should initially be zero.

We can verify this numerically.


In [ ]:
with torch.no_grad():
    base = x @ layer.weight.T
    lora_branch = (
        layer.alpha / layer.rank
    ) * (x @ layer.A.T) @ layer.B.T

print("Base output norm :", base.norm().item())
print("LoRA branch norm :", lora_branch.norm().item())
print("Total output norm:", layer(x).norm().item())


Base output norm : 7.120668888092041
LoRA branch norm : 0.0
Total output norm: 7.120668888092041


# 11. Gradient Flow: What Actually Gets Trained?

Suppose the loss is

$$
\mathcal{L}.
$$

The original parameter $W$ is frozen.

Therefore we do not update $W$.

The trainable parameters are:

$$
A
\quad\text{and}\quad
B.
$$

The optimization problem becomes

$$
\min_{A,B}
\mathcal{L}
\left(
W+
\frac{\alpha}{r}BA
\right),
$$

where $W$ is treated as fixed.

### Important concept

The frozen weight $W$ is still used in the forward pass.

So:

$$
\boxed{
\text{frozen} \neq \text{removed}
}
$$

Frozen means its value is not updated by the optimizer.


In [ ]:
target = torch.randn(3, 4)

optimizer = torch.optim.AdamW(
    [p for p in layer.parameters() if p.requires_grad],
    lr=1e-2
)

optimizer.zero_grad()

loss = ((layer(x) - target) ** 2).mean()
loss.backward()

print("Loss:", loss.item())
print("W gradient:", layer.weight.grad)
print("A gradient norm:", layer.A.grad.norm().item())
print("B gradient norm:", layer.B.grad.norm().item())


Loss: 4.903493881225586
W gradient: None
A gradient norm: 0.0
B gradient norm: 0.0774325579404831


# 12. The Role of Rank $r$

The rank determines the capacity of the LoRA update.

The number of trainable parameters is

$$
P_{LoRA}
=
r(d_{in}+d_{out}).
$$

Therefore increasing $r$ increases the number of trainable parameters.

For example:

$$
r=4
$$

has less capacity than

$$
r=32.
$$

However, a larger rank does **not** mathematically guarantee better validation performance.

Rank is a capacity hyperparameter, and its effect must be evaluated experimentally.


# 13. The Role of the Scaling Factor $\alpha$

LoRA uses

$$
W' =
W+
\frac{\alpha}{r}BA.
$$

The effective scaling factor is

$$
s=\frac{\alpha}{r}.
$$

For example, if

$$
r=8, \qquad \alpha=16,
$$

then

$$
s=2.
$$

If

$$
r=8, \qquad \alpha=8,
$$

then

$$
s=1.
$$

### Rank versus alpha

Changing $r$ changes:

- adapter capacity,
- number of trainable parameters,
- maximum possible rank of $BA$.

Changing $\alpha$ changes:

- the scale of the LoRA contribution.

Changing $\alpha$ alone does not change the number of LoRA parameters.


In [ ]:
# Demonstrate the effect of alpha on the LoRA branch

torch.manual_seed(0)

d_in, d_out, r = 16, 12, 2

x = torch.randn(8, d_in)
A = torch.randn(r, d_in) * 0.01
B = torch.randn(d_out, r) * 0.01

for alpha in [1, 4, 8, 16, 32]:
    update = (
        alpha / r
    ) * (x @ A.T) @ B.T

    print(
        f"alpha={alpha:2d} | "
        f"update norm={update.norm().item():.6f}"
    )


alpha= 1 | update norm=0.001740
alpha= 4 | update norm=0.006961
alpha= 8 | update norm=0.013923
alpha=16 | update norm=0.027845
alpha=32 | update norm=0.055691


# 14. LoRA Dropout

LoRA implementations can apply dropout to the LoRA branch.

Conceptually:

$$
x
\rightarrow A
\rightarrow \operatorname{Dropout}
\rightarrow B.
$$

The purpose is regularization.

The base branch is still

$$
Wx.
$$

The complete computation remains conceptually:

$$
Wx+
\frac{\alpha}{r}
B\,\operatorname{Dropout}(Ax).
$$

The exact implementation details depend on the PEFT library and configuration.


# 15. Where Is LoRA Applied in a Transformer?

A Transformer contains many linear transformations.

For self-attention, a simplified formulation is

$$
Q=XW_Q,
$$

$$
K=XW_K,
$$

$$
V=XW_V.
$$

There is also an output projection:

$$
O=ZW_O.
$$

LoRA can be attached to selected linear projections.

Common target names in different Transformer architectures include:

- `q_proj`
- `k_proj`
- `v_proj`
- `o_proj`
- `q_lin`
- `v_lin`

The names are **architecture-dependent**.

Therefore, before configuring LoRA, inspect the model's module names.


# 16. Why Target-Module Selection Matters

Suppose LoRA is applied only to query and value projections.

Then only those transformations receive trainable low-rank corrections.

If LoRA is applied to many attention and MLP projections, the number of trainable parameters increases.

Thus target selection controls where the model is allowed to adapt.

There is no single target-module configuration that is guaranteed to be optimal for every model and task.


# 17. From Manual LoRA to Hugging Face PEFT

For the practical experiment we will use:

- **PyTorch**
- **Transformers**
- **Datasets**
- **PEFT**

We will fine-tune a lightweight pretrained Transformer for binary sentiment classification.

### Installation

Run the following in a terminal or notebook cell if required:

```bash
pip install -U torch transformers datasets peft accelerate evaluate
```

The manual LoRA sections above only require PyTorch.


In [ ]:
import torch
import torch.nn as nn

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)

print("All imports successful.")
%pip install -U torch==2.2.2 transformers datasets peft accelerate evaluate "torchao>=0.16.0"

All imports successful.
ERROR: Could not find a version that satisfies the requirement torch==2.2.2 (from versions: 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0, 2.12.1, 2.13.0, 2.14.0)
ERROR: No matching distribution found for torch==2.2.2


In [ ]:
import torch
import torch.nn as nn

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)

print("All imports successful.")


All imports successful.


# 18. Load a Pretrained Transformer

We use `distilbert-base-uncased` because it is relatively lightweight and convenient for a classroom demonstration.

The model will perform binary sentiment classification on IMDb reviews.


In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print("Model loaded.")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.


In [ ]:
# Inspect linear layers

for name, module in base_model.named_modules():
    if isinstance(module, nn.Linear):
        print(
            name,
            "->",
            module.in_features,
            "x",
            module.out_features
        )


distilbert.transformer.layer.0.attention.q_lin -> 768 x 768
distilbert.transformer.layer.0.attention.k_lin -> 768 x 768
distilbert.transformer.layer.0.attention.v_lin -> 768 x 768
distilbert.transformer.layer.0.attention.out_lin -> 768 x 768
distilbert.transformer.layer.0.ffn.lin1 -> 768 x 3072
distilbert.transformer.layer.0.ffn.lin2 -> 3072 x 768
distilbert.transformer.layer.1.attention.q_lin -> 768 x 768
distilbert.transformer.layer.1.attention.k_lin -> 768 x 768
distilbert.transformer.layer.1.attention.v_lin -> 768 x 768
distilbert.transformer.layer.1.attention.out_lin -> 768 x 768
distilbert.transformer.layer.1.ffn.lin1 -> 768 x 3072
distilbert.transformer.layer.1.ffn.lin2 -> 3072 x 768
distilbert.transformer.layer.2.attention.q_lin -> 768 x 768
distilbert.transformer.layer.2.attention.k_lin -> 768 x 768
distilbert.transformer.layer.2.attention.v_lin -> 768 x 768
distilbert.transformer.layer.2.attention.out_lin -> 768 x 768
distilbert.transformer.layer.2.ffn.lin1 -> 768 x 3072
dist

# 19. Prepare the IMDb Dataset

For a classroom run, we use a subset of the dataset.

The dataset contains:

- review text
- binary sentiment label

We tokenize the reviews and truncate them to a maximum length.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

train_ds = (
    dataset["train"]
    .shuffle(seed=42)
    .select(range(2000))
)

eval_ds = (
    dataset["test"]
    .shuffle(seed=42)
    .select(range(500))
)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256
    )

train_ds = train_ds.map(
    tokenize,
    batched=True
)

eval_ds = eval_ds.map(
    tokenize,
    batched=True
)

train_ds = train_ds.rename_column(
    "label",
    "labels"
)

eval_ds = eval_ds.rename_column(
    "label",
    "labels"
)

train_ds.set_format("torch")
eval_ds.set_format("torch")

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print(train_ds)
print(eval_ds)


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})
Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 500
})


# 20. Configure LoRA

For DistilBERT, the attention projection modules include `q_lin` and `v_lin`.

We will attach LoRA to these modules.

The configuration is:

$$
r=8
$$

and

$$
\alpha=16.
$$

Therefore,

$$
\frac{\alpha}{r}=2.
$$

The adapter dropout is set to $0.1$.


In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    bias="none",
)

lora_model = get_peft_model(
    base_model,
    lora_config
)

lora_model.print_trainable_parameters()


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


# 21. Verify Trainable Parameters

The most important property to verify is:

$$
\boxed{
\text{Base model parameters are frozen}
}
$$

while

$$
\boxed{
\text{LoRA parameters are trainable}
}
$$

We can calculate the total and trainable parameter counts directly.


In [ ]:
total_params = sum(
    p.numel()
    for p in lora_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in lora_model.parameters()
    if p.requires_grad
)

percentage = (
    100 * trainable_params / total_params
)

print("Total parameters     :", f"{total_params:,}")
print("Trainable parameters :", f"{trainable_params:,}")
print("Trainable percentage :", f"{percentage:.4f}%")


Total parameters     : 67,694,596
Trainable parameters : 739,586
Trainable percentage : 1.0925%


In [ ]:
# Show trainable parameters

for name, parameter in lora_model.named_parameters():
    if parameter.requires_grad:
        print(
            name,
            tuple(parameter.shape)
        )


base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_A.default.weight (8, 768)
base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_B.default.weight (768, 8)
base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_A.default.weight (8, 768)
base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_B.default.weight (768, 8)
base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_A.default.weight (8, 768)
base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_B.default.weight (768, 8)
base_model.model.distilbert.transformer.layer.1.attention.v_lin.lora_A.default.weight (8, 768)
base_model.model.distilbert.transformer.layer.1.attention.v_lin.lora_B.default.weight (768, 8)
base_model.model.distilbert.transformer.layer.2.attention.q_lin.lora_A.default.weight (8, 768)
base_model.model.distilbert.transformer.layer.2.attention.q_lin.lora_B.default.weight (768, 8)
base_model.model.distilbert.transformer.layer.2.at

# 22. Training Configuration

We now create a Hugging Face `Trainer`.

The learning rate used here is larger than a typical full fine-tuning learning rate because only a small adapter is being trained. This is an example configuration, not a universal optimum.


In [ ]:
training_args = TrainingArguments(
    output_dir="./lora-distilbert-imdb",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Trainer created.")


Trainer created.


# 23. Train the LoRA Adapter

The next cell performs actual fine-tuning.

Only the LoRA parameters are trainable.

The pretrained Transformer remains frozen.


In [ ]:
# Run this cell to train.
train_result = trainer.train()

print(train_result)


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  self._dataset_fetcher = _DatasetKind.create_fetcher(
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,0.449845,0.323441
2,0.309019,0.314959


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  self._dataset_fetcher = _DatasetKind.create_fetcher(
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  self._dataset_fetcher = _DatasetKind.create_fetcher(


TrainOutput(global_step=250, training_loss=0.4041481285095215, metrics={'train_runtime': 5255.0986, 'train_samples_per_second': 0.761, 'train_steps_per_second': 0.048, 'total_flos': 269478813696000.0, 'train_loss': 0.4041481285095215, 'epoch': 2.0})


# 24. Evaluate the Fine-Tuned Model


In [ ]:
metrics = trainer.evaluate()

for key, value in metrics.items():
    print(f"{key}: {value}")


# 25. Inspect the Learned LoRA Parameters

After training, the adapter matrices have learned task-specific information.

The LoRA parameters are typically exposed with names containing `lora_A` and `lora_B`.


In [ ]:
for name, parameter in lora_model.named_parameters():
    if parameter.requires_grad and (
        "lora_A" in name or
        "lora_B" in name
    ):
        print(
            f"{name:80s} "
            f"norm={parameter.detach().norm().item():.6f}"
        )


# 26. Save the LoRA Adapter

One practical advantage of LoRA is that the task-specific adapter can be stored separately from the base model.

Conceptually, we have:

$$
\boxed{
\text{Base Model} + \text{LoRA Adapter}
}
$$

instead of storing a completely separate full model for every task.


In [ ]:
ADAPTER_PATH = "./distilbert-imdb-lora"

lora_model.save_pretrained(
    ADAPTER_PATH
)

tokenizer.save_pretrained(
    ADAPTER_PATH
)

print("Adapter saved to:", ADAPTER_PATH)


# 27. Reload the Adapter

The base model and adapter can be loaded separately.

This is useful when the same base model is shared by several task-specific adapters.

For example:

$$
\text{Base}
+
\text{Sentiment Adapter}
$$

and

$$
\text{Base}
+
\text{Domain Adapter}.
$$


In [ ]:
from peft import PeftModel

fresh_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

reloaded_model = PeftModel.from_pretrained(
    fresh_base,
    ADAPTER_PATH
)

reloaded_model.print_trainable_parameters()


# 28. Merging the LoRA Adapter

Recall the mathematical equation:

$$
W_{effective}
=
W+
\frac{\alpha}{r}BA.
$$

The adapter can optionally be merged into the base weights.

In PEFT this can be done with:

```python
merged_model = lora_model.merge_and_unload()
```

After merging, the resulting model can be used like a normal model.

### Why merge?

Potential advantages:

- simpler deployment,
- no separate adapter branch,
- convenient inference.

### Why keep the adapter separate?

Potential advantages:

- multiple task adapters can share one base model,
- adapters are small,
- tasks can be switched more easily.


In [ ]:
# Optional: merge after training

# merged_model = lora_model.merge_and_unload()
# print(type(merged_model))


# 29. LoRA Parameter Count: A Concrete Exercise

Consider a weight matrix

$$
W\in\mathbb{R}^{4096\times4096}.
$$

### Full fine-tuning

$$
P_{full}
=
4096\times4096.
$$

### LoRA with $r=8$

$$
P_{LoRA}
=
8(4096+4096).
$$

### Questions

1. Calculate both parameter counts.
2. Calculate the percentage of parameters trained by LoRA.
3. If the same LoRA configuration is applied to 32 such matrices, calculate the total trainable LoRA parameters.
4. Explain why the parameter savings grow dramatically for large matrices.


In [ ]:
d_in = 4096
d_out = 4096
r = 8
num_matrices = 32

full_one = d_in * d_out
lora_one = r * (d_in + d_out)

full_total = num_matrices * full_one
lora_total = num_matrices * lora_one

print("Full parameters for one matrix :", f"{full_one:,}")
print("LoRA parameters for one matrix :", f"{lora_one:,}")
print("Full parameters for 32 matrices:", f"{full_total:,}")
print("LoRA parameters for 32 matrices:", f"{lora_total:,}")
print(
    "LoRA percentage:",
    f"{100*lora_total/full_total:.4f}%"
)


# 30. Experiment: Change the Rank

Try:

$$
r \in \{2,4,8,16\}.
$$

Keep all other settings fixed.

Record:

| Rank | Trainable Parameters | Validation Metric | Training Time |
|---:|---:|---:|---:|
| 2 | | | |
| 4 | | | |
| 8 | | | |
| 16 | | | |

### Questions

1. How does trainable parameter count change?
2. Does doubling $r$ double the LoRA parameter count?
3. Does increasing $r$ guarantee better validation performance?
4. What trade-off exists between adaptation capacity and parameter efficiency?


# 31. Experiment: Change $\alpha$

Keep

$$
r=8
$$

fixed and compare:

$$
\alpha \in \{4,8,16,32\}.
$$

For each experiment, record:

- training loss,
- validation performance,
- training time,
- trainable parameter count.

### Reasoning questions

1. Does changing $\alpha$ change the number of trainable parameters?
2. Does changing $\alpha$ change the maximum rank of $BA$?
3. What happens to the scale $\alpha/r$?
4. Why can changing the scale affect optimization even though the adapter architecture remains unchanged?


# 32. LoRA Versus Full Fine-Tuning

| Property | Full Fine-Tuning | LoRA |
|---|---|---|
| Base weights trainable | Yes | No |
| Additional matrices | No | Yes |
| Trainable parameter count | Large | Small |
| Task-specific checkpoint | Large | Small adapter |
| Update restriction | Unrestricted | Low-rank |
| Base model reused | Less convenient | Convenient |
| Mathematical update | $W+\Delta W$ | $W+\frac{\alpha}{r}BA$ |

The key mathematical difference is:

### Full fine-tuning

$$
W' = W + \Delta W
$$

where $\Delta W$ is unrestricted.

### LoRA

$$
W' =
W+
\frac{\alpha}{r}BA
$$

where

$$
\operatorname{rank}(BA)\leq r.
$$


# 33. Common Misconceptions

### Misconception 1

**"LoRA slightly changes the original pretrained weights."**

Usually, no. The standard LoRA setup freezes the original weights and learns separate adapter parameters.

### Misconception 2

**"LoRA removes the base model."**

No. The base model is still used during the forward pass.

### Misconception 3

**"Rank $r$ is the rank of the pretrained matrix."**

No. It controls the maximum rank of the learned update $BA$.

### Misconception 4

**"A larger rank always gives a better model."**

Not necessarily. A larger rank gives greater adapter capacity, but task performance must be evaluated experimentally.

### Misconception 5

**"LoRA is simply factorizing the pretrained weight matrix."**

No. LoRA factorizes the **learned update**:

$$
\Delta W=BA.
$$

### Misconception 6

**"Frozen parameters do not participate in computation."**

They still participate in the forward pass. They simply are not updated by the optimizer.


# 34. Final Mathematical Summary

The original linear transformation is

$$
y=Wx.
$$

LoRA changes it to

$$
\boxed{
y=
Wx+
\frac{\alpha}{r}BAx
}
$$

with

$$
A\in\mathbb{R}^{r\times d_{in}},
$$

and

$$
B\in\mathbb{R}^{d_{out}\times r}.
$$

The effective weight is

$$
\boxed{
W_{effective}
=
W+
\frac{\alpha}{r}BA
}
$$

The trainable parameter count is

$$
\boxed{
P_{LoRA}
=
r(d_{in}+d_{out})
}
$$

instead of

$$
\boxed{
P_{full}
=
d_{in}d_{out}
}
$$

and

$$
\boxed{
\operatorname{rank}(\Delta W)\leq r.
}
$$

---


